## Import Libraries

In [4]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    StandardScaler
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    ConfusionMatrixDisplay
)
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

## Load Feature Engineered Dataset

In [5]:
DATA_PATH = r"D:\Telco_Customer_Churn\data\processed_data\churn_feature_engineered.csv"
FIGURES_DIR = r"D:\Telco_Customer_Churn\reports\figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(3)

Dataset loaded: 7043 rows x 25 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TotalServicesSubscribed,CustomerTenureGroup,CustomerValueTier,DigitalAdoptionScore
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,2,New Customer,Low Value,2
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,One year,No,Mailed check,56.95,1889.50,No,4,Regular Customer,Medium Value,1
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,4,New Customer,Low Value,3


# Dataset Overview

Understand the dataset before model training.

We will inspect:

- Dataset Shape
- Sample Records
- Data Types
- Missing Values

In [6]:
# Shape
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

# First five rows
df.head()

Rows    : 7043
Columns : 25


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TotalServicesSubscribed,CustomerTenureGroup,CustomerValueTier,DigitalAdoptionScore
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,2,New Customer,Low Value,2
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,One year,No,Mailed check,56.95,1889.50,No,4,Regular Customer,Medium Value,1
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,4,New Customer,Low Value,3
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,One year,No,Bank transfer (automatic),42.30,1840.75,No,4,Regular Customer,Medium Value,1
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,2,New Customer,Low Value,1


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerID               7043 non-null   str    
 1   gender                   7043 non-null   str    
 2   SeniorCitizen            7043 non-null   int64  
 3   Partner                  7043 non-null   str    
 4   Dependents               7043 non-null   str    
 5   tenure                   7043 non-null   int64  
 6   PhoneService             7043 non-null   str    
 7   MultipleLines            7043 non-null   str    
 8   InternetService          7043 non-null   str    
 9   OnlineSecurity           7043 non-null   str    
 10  OnlineBackup             7043 non-null   str    
 11  DeviceProtection         7043 non-null   str    
 12  TechSupport              7043 non-null   str    
 13  StreamingTV              7043 non-null   str    
 14  StreamingMovies          7043 non-n

In [8]:
1
df.isnull().sum()

customerID                 0
gender                     0
SeniorCitizen              0
Partner                    0
Dependents                 0
tenure                     0
PhoneService               0
MultipleLines              0
InternetService            0
OnlineSecurity             0
OnlineBackup               0
DeviceProtection           0
TechSupport                0
StreamingTV                0
StreamingMovies            0
Contract                   0
PaperlessBilling           0
PaymentMethod              0
MonthlyCharges             0
TotalCharges               0
Churn                      0
TotalServicesSubscribed    0
CustomerTenureGroup        0
CustomerValueTier          0
DigitalAdoptionScore       0
dtype: int64

## Review Engineered Features

In [9]:
engineered_features = [
    'TotalServicesSubscribed',
    'CustomerTenureGroup',
    'CustomerValueTier',
    'DigitalAdoptionScore'
]

df[engineered_features].head()

,TotalServicesSubscribed,CustomerTenureGroup,CustomerValueTier,DigitalAdoptionScore
0,2,New Customer,Low Value,2
1,4,Regular Customer,Medium Value,1
2,4,New Customer,Low Value,3
3,4,Regular Customer,Medium Value,1
4,2,New Customer,Low Value,1


## Feature Selection

In [10]:
# Remove identifier column
df = df.drop(columns=['customerID'])

In [11]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TotalServicesSubscribed,CustomerTenureGroup,CustomerValueTier,DigitalAdoptionScore
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,2,New Customer,Low Value,2
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,One year,No,Mailed check,56.95,1889.50,No,4,Regular Customer,Medium Value,1
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,4,New Customer,Low Value,3
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,One year,No,Bank transfer (automatic),42.30,1840.75,No,4,Regular Customer,Medium Value,1
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,2,New Customer,Low Value,1


## Separate Features and Target

In [12]:
# Features
X = df.drop(columns=['Churn'])

# Target
y = df['Churn']

In [13]:
print("Features Shape :", X.shape)
print("Target Shape   :", y.shape)

Features Shape : (7043, 23)
Target Shape   : (7043,)


In [14]:
print(X.columns.tolist())

['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'TotalServicesSubscribed', 'CustomerTenureGroup', 'CustomerValueTier', 'DigitalAdoptionScore']


In [15]:
y.value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

## Train-Test Split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
                                                X,
                                                y,
                                                test_size=0.2,
                                                random_state=42,
                                                stratify=y
                                            )

In [18]:
print(f"X train    : {X_train.shape}")
print(f"X test : {X_test.shape}")
print(f"Y Train    : {y_train.shape}")
print(f"Y test : {y_test.shape}")

X train    : (5634, 23)
X test : (1409, 23)
Y Train    : (5634,)
Y test : (1409,)


In [21]:
print(f"Y original  :  {y.value_counts()}")
print(f"Y train  :  {y_train.value_counts()}")
print(f"Y test  :  {y_test.value_counts()}")

Y original  :  Churn
No     5174
Yes    1869
Name: count, dtype: int64
Y train  :  Churn
No     4139
Yes    1495
Name: count, dtype: int64
Y test  :  Churn
No     1035
Yes     374
Name: count, dtype: int64
